# Noble AI — Laboratorio integral Colab v1.4

Ejecuta **Entorno de ejecución → Ejecutar todas**. Este notebook clonará Noble AI, instalará dependencias, compilará frontend, ejecutará tests y, cuando haya NVIDIA/CUDA, probará un GGUF auténtico.

**Diagnóstico visible:** cada etapa enseña hora de inicio, estado, últimas líneas de salida, tiempo transcurrido y avisos cada 12 segundos cuando un comando no imprime nada. Cada comando tiene un límite de ejecución. Se genera un ZIP con resultados incluso cuando fallan etapas individuales.

La app solo escucha en localhost de Colab. No se publican túneles ni puertos públicos. El notebook se puede ejecutar sin GPU (la inferencia quedará SKIP). Si detuviste el notebook anterior tras 30 minutos, puedes usar este nuevo cuaderno sin depender del archivo descargable anterior.

**v1.4:** corrige el HTTP 404 del ensayo v1.3; comprueba el catálogo real de Hugging Face y registra la revisión/huella SHA256 del GGUF antes de comenzar las generaciones. Los resultados v1.3 se mantienen separados en su carpeta histórica.


In [ ]:
RUN_GPU = True             # GPU real con modelo GGUF, si está disponible
RUN_BROWSER = True         # Playwright escritorio/móvil
RUNS = 3                   # 1 a 10 generaciones narrativas
KEEP_APP_RUNNING = True
MODEL_REPO = "bartowski/Qwen_Qwen3-4B-Instruct-2507-GGUF"
MODEL_FILE = "Qwen_Qwen3-4B-Instruct-2507-Q4_K_M.gguf"
API_PORT = 8181            # Evita conflicto con el notebook anterior
LLM_PORT = 5181

from pathlib import Path
import os, sys, json, time, subprocess, shutil, zipfile, re, threading, traceback
from datetime import datetime, timezone

BASE = Path("/content/noble_qa_v14")
ROOT = BASE / "novel_ai"
EVIDENCE = BASE / "diagnostico"
LOGS = EVIDENCE / "logs"
ZIP_FILE = Path("/content/Noble_AI_Colab_QA_v1_4.zip")
LOGS.mkdir(parents=True, exist_ok=True)
T0 = time.monotonic()
state = {"version": "1.3", "stages": {}, "environment": {}, "benchmarks": {},
         "started_utc": datetime.now(timezone.utc).isoformat()}
children = {}

def clock():
    return datetime.now(timezone.utc).strftime("%H:%M:%S UTC")
def clean(value):
    text = str(value)
    text = re.sub(r"(?i)(Bearer\s+)[^\s\"']+", r"\1[REDACTED]", text)
    text = re.sub(r"(?i)((?:api[_-]?key|password|secret|token)\s*[:=]\s*)[^\s\"']+", r"\1[REDACTED]", text)
    return text
def save_status():
    state["updated_utc"] = datetime.now(timezone.utc).isoformat()
    state["elapsed_seconds"] = round(time.monotonic() - T0, 1)
    (EVIDENCE / "report.json").write_text(json.dumps(state, indent=2, ensure_ascii=False), encoding="utf-8")
    lines = [f"{name} — {result['status']}: {result.get('detail','')}"
             for name, result in state["stages"].items()]
    (EVIDENCE / "ESTADO_ACTUAL.txt").write_text(
        f"{clock()} | {state['elapsed_seconds']} segundos\n" + "\n".join(lines) + "\n",
        encoding="utf-8")
def update(name, status, detail=""):
    state["stages"][name] = {"status": status, "detail": clean(detail)[:360],
                             "updated_utc": clock()}
    print(f"[{clock()}] [{status}] {name}: {clean(detail)[:240]}", flush=True)
    save_status()
def skip(name, why):
    update(name, "SKIP", why)

def command(name, argv, cwd=None, timeout=600, environment=None, watch=None):
    """Logs en directo. Nunca capture_output para instalaciones largas."""
    update(name, "RUNNING", f"Ejecutando (límite {timeout}s): " + " ".join(argv[:3]))
    logfile = LOGS / (name + ".log")
    began = time.monotonic()
    last = began
    offset = 0
    fragment = ""
    proc = None
    try:
        with logfile.open("wb") as sink:
            proc = subprocess.Popen(argv, cwd=str(cwd or BASE), env=environment,
                                    stdout=sink, stderr=subprocess.STDOUT,
                                    stdin=subprocess.DEVNULL, start_new_session=True)
            while proc.poll() is None:
                time.sleep(2)
                now = time.monotonic()
                with logfile.open("rb") as reader:
                    reader.seek(offset)
                    chunk = reader.read(200000)
                    offset = reader.tell()
                if chunk:
                    fragment += chunk.decode("utf-8", errors="replace")
                    lines = fragment.splitlines(keepends=True)
                    fragment = "" if not lines or lines[-1].endswith(("\n","\r")) else lines.pop()
                    latest = [clean(line.strip()) for line in lines if line.strip()]
                    for line in latest[-4:]:
                        print("   [" + name + "] " + line[:240], flush=True)
                    if latest: last = now
                if now - last >= 12:
                    extra = ""
                    if watch and Path(watch).exists():
                        try:
                            amount = sum(f.stat().st_size for f in Path(watch).rglob("*") if f.is_file())
                            extra = f" | archivos ~{amount/1048576:.1f} MB"
                        except OSError: pass
                    print(f"   [ACTIVO] {name}: {int(now-began)}s de {timeout}s límite{extra}", flush=True)
                    state["stages"][name]["running_seconds"] = round(now-began, 1)
                    save_status()
                    last = now
                if now - began >= timeout:
                    try:
                        import signal
                        os.killpg(proc.pid, signal.SIGTERM)
                        proc.wait(timeout=5)
                    except (OSError, subprocess.TimeoutExpired):
                        proc.kill(); proc.wait()
                    update(name, "FAIL", f"Se superó el límite de {timeout}s. Log: {logfile.name}")
                    return False
        if fragment.strip(): print("   ["+name+"] "+clean(fragment.strip())[:240], flush=True)
        ok = proc.returncode == 0
        if not ok:
            for line in logfile.read_text(errors="replace").splitlines()[-8:]:
                print("   [ERROR] " + clean(line)[:230], flush=True)
        update(name, "PASS" if ok else "FAIL",
               f"Código {proc.returncode}, duración {int(time.monotonic()-began)}s, log {logfile.name}")
        return ok
    except Exception as error:
        update(name, "FAIL", type(error).__name__ + ": " + str(error)[:190])
        return False
    finally:
        if proc is not None and proc.poll() is None:
            proc.terminate()
        if logfile.exists():
            try:
                with logfile.open("rb") as src:
                    src.seek(max(0, logfile.stat().st_size - 450000))
                    data = src.read().decode(errors="replace")
                logfile.write_text(clean(data), encoding="utf-8")
            except OSError: pass

def gpu_snapshot():
    if not shutil.which("nvidia-smi"): return []
    try:
        proc = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,memory.used,utilization.gpu",
                        "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=7, check=True)
        result = []
        for line in proc.stdout.splitlines():
            parts = [part.strip() for part in line.split(",")]
            if len(parts)==4:
                result.append({"name":parts[0],"total_mb":int(parts[1]),
                   "used_mb":int(parts[2]),"util_percent":int(parts[3])})
        return result
    except Exception: return []

def http_json(url, data=None, method="GET", timeout=10):
    import urllib.request
    raw = json.dumps(data).encode() if data is not None else None
    req = urllib.request.Request(url, data=raw, method=method,
          headers={"Content-Type":"application/json"} if raw is not None else {})
    with urllib.request.urlopen(req, timeout=timeout) as response:
        return json.load(response)

save_status()
print("Noble AI QA v1.4",clock(),"· Registro de cada etapa y aviso cada 12 segundos.",flush=True)

## 1. Descargar Noble AI y reconocer el hardware

El estado **PASS/FAIL** se muestra en esta celda. Los logs se guardan en `diagnostico/logs/`.

In [ ]:
if (ROOT / ".git").is_dir():
    REPO_OK = command("git_update", ["git","-C",str(ROOT),"pull","--ff-only"], timeout=240)
else:
    REPO_OK = command("git_clone", ["git","clone","--progress","--depth","1",
          "https://github.com/Mairon31/novel_ai.git",str(ROOT)], timeout=480)
if REPO_OK:
    version = subprocess.run(["git","-C",str(ROOT),"rev-parse","HEAD"],
                             capture_output=True,text=True,timeout=12)
    state["environment"]["commit"] = version.stdout.strip()
    update("git_revision","PASS",version.stdout.strip())
else:
    skip("git_revision","Sin repositorio disponible")
state["environment"]["python"] = sys.version.split()[0]
state["environment"]["gpu_start"] = gpu_snapshot()
state["environment"]["disk_free_gib"] = round(shutil.disk_usage(BASE).free/1024**3,2)
save_status()
print("Hardware NVIDIA detectado:", state["environment"]["gpu_start"], flush=True)

## 2. Instalar dependencias, compilar y ejecutar los tests Python

Las pruebas y la compilación se ejecutan con límites de tiempo independientes. Si una etapa falla, se registra el problema y continúa con lo que pueda verificarse.

In [ ]:
PY_READY = False
WEB_READY = False
if REPO_OK:
    PY_READY = command("python_deps",
        [sys.executable,"-m","pip","install","-r",str(ROOT/"apps/api/requirements.txt")],
        timeout=900)
    if PY_READY:
        command("python_tests",[sys.executable,"-u","-m","unittest","discover",
           "-s","tests","-p","test_*.py","-v"],cwd=ROOT,timeout=480)
    else:
        skip("python_tests","Falló la instalación Python")
    try:
        node = subprocess.run(["node","--version"],capture_output=True,text=True,timeout=8)
        major = int(node.stdout.strip().lstrip("v").split(".")[0])
        NODE_READY = major>=22
    except Exception:
        NODE_READY = False
    if not NODE_READY:
        if command("node_manager",["npm","install","-g","n","--no-audit","--no-fund"],timeout=360):
            NODE_READY = command("node_22",["n","22"],timeout=480)
    if NODE_READY:
        web = ROOT/"apps/web"
        dependencies = command("web_deps",["npm","install","--no-audit","--no-fund"],cwd=web,timeout=900)
        if dependencies:
            WEB_READY = command("web_build",["npm","run","build"],cwd=web,timeout=600)
        else:
            skip("web_build","Fallaron las dependencias web")
    else:
        skip("web_build","Node.js 22 no disponible")
else:
    for stage in ("python_deps","python_tests","web_deps","web_build"):
        skip(stage,"Falló la obtención del repositorio")
save_status()

## 3. Desplegar Noble AI en el entorno Colab

El backend sirve la compilación de React por `127.0.0.1:8181`. Su arranque tiene un límite de espera y guarda logs del servidor, en lugar de bloquear la ejecución sin explicación.

In [ ]:
API_READY = False
if PY_READY:
    server_log = (LOGS/"noble_api.log").open("ab", buffering=0)
    environment = os.environ.copy()
    environment["NOBLE_DATA_DIR"] = str(BASE/"workspace_de_prueba")
    children["api"] = subprocess.Popen(
        [sys.executable,"-u","-m","uvicorn","noble_api.main:app",
         "--app-dir",str(ROOT/"apps/api"),"--host","127.0.0.1","--port",str(API_PORT)],
        cwd=str(ROOT),env=environment,stdout=server_log,
        stderr=subprocess.STDOUT,stdin=subprocess.DEVNULL,start_new_session=True)
    update("api_deploy","RUNNING","Esperando /health hasta 90 segundos")
    for i in range(90):
        try:
            health = http_json(f"http://127.0.0.1:{API_PORT}/api/v1/health",timeout=3)
            if health.get("status") == "ok":
                API_READY = True
                break
        except Exception: pass
        if children["api"].poll() is not None:
            break
        if i%12==11:
            print(f"   [ACTIVO] FastAPI {i+1}s | proceso {children['api'].poll()} | log noble_api.log",flush=True)
        time.sleep(1)
    update("api_deploy","PASS" if API_READY else "FAIL",
           "Servidor HTTP local listo" if API_READY else "No respondió: revisar noble_api.log")
else:
    skip("api_deploy","Dependencias Python no disponibles")
if API_READY and WEB_READY:
    try:
        import urllib.request
        with urllib.request.urlopen(f"http://127.0.0.1:{API_PORT}/",timeout=10) as r:
            html = r.read(5000).decode(errors="replace").lower()
        update("ui_deploy","PASS" if "<html" in html else "FAIL",
               "React compilado servido por FastAPI")
    except Exception as exc:
        update("ui_deploy","FAIL",type(exc).__name__+": "+str(exc)[:150])
else:
    skip("ui_deploy","API o frontend no disponible")

## 4. QA visual de escritorio y móvil

Instala Chromium y ejecuta las pruebas de Playwright contra la versión compilada. El resultado incluye archivos de `test-results` cuando estén disponibles.

In [ ]:
if RUN_BROWSER and API_READY and WEB_READY:
    web = ROOT/"apps/web"
    chromium = command("playwright_install",
        ["npx","playwright","install","chromium","--with-deps"],
        cwd=web,timeout=1200)
    if chromium:
        environment = os.environ.copy()
        environment["NOBLE_E2E_BASE_URL"] = f"http://127.0.0.1:{API_PORT}"
        command("browser_tests",["npm","run","test:e2e"],
                cwd=web,timeout=900,environment=environment)
        source = web/"test-results"
        if source.is_dir():
            shutil.copytree(source,EVIDENCE/"browser-artifacts",dirs_exist_ok=True)
    else:
        skip("browser_tests","No se instaló Playwright Chromium")
else:
    skip("browser_tests","Desactivado o despliegue incompleto")
save_status()

## 5. GPU real: binarios oficiales CUDA, GGUF y benchmark narrativo

Esta versión evita compilar CUDA desde cero. Utiliza `llama.cpp` precompilado b12119 (CUDA 12.8) y comprueba **SHA256 de los dos archivos descargados**, dependencias dinámicas, carga del modelo y tres generaciones reales. Los logs muestran progreso y errores. Si el binario no es compatible con Colab, la etapa falla explícitamente; no se presenta una prueba falsa.

In [ ]:
GPU_RESULT = False
if not RUN_GPU:
    skip("gpu_benchmark","Prueba GPU desactivada")
elif not API_READY:
    skip("gpu_benchmark","No está disponible la API Noble AI")
elif not gpu_snapshot():
    skip("gpu_benchmark","No hay GPU NVIDIA: activa GPU en el entorno de ejecución")
else:
    try:
        # Official pre-built CUDA 12.8 distribution pinned to release b12119.
        # Previous notebook spent 30 minutes recompiling hundreds of nvcc kernels.
        engine_repo = BASE/"llama.cpp"
        binaries_dir = engine_repo/"build"/"bin"
        binaries_dir.mkdir(parents=True,exist_ok=True)
        installer = r'''
import hashlib,os,shutil,sys,tarfile,urllib.request,time
from pathlib import Path

root=Path(sys.argv[1])
bin_dir=Path(sys.argv[2])
release="b12119"
root.mkdir(parents=True,exist_ok=True)
bin_dir.mkdir(parents=True,exist_ok=True)
assets=[
 ("llama-b12119-bin-ubuntu-cuda-12.8-x64.tar.gz",
  "d61b6c35f56f3cf74ca8a0ec5bdd8c8dd1b5c304f9e001bf9ddb29f8d4d2a409"),
 ("cudart-llama-b12119-bin-ubuntu-cuda-12.8-x64.tar.gz",
  "40ddc5e887003a5ac19f1baf0681d9bedb52bf7cf57b9fa896dbaf6439e53b7c"),
]
base_url="https://github.com/ggml-org/llama.cpp/releases/download/"+release+"/"
for name,expected_sha in assets:
    dest=root/name
    if dest.exists():
        h=hashlib.sha256()
        with dest.open("rb") as stream:
            for block in iter(lambda: stream.read(8*1024*1024),b""):h.update(block)
        if h.hexdigest()!=expected_sha:
            print("Caché con hash incorrecto, volviendo a descargar:",name,flush=True)
            dest.unlink()
    if not dest.exists():
        print("DESCARGA INICIADA:",name,flush=True)
        temp=dest.with_suffix(dest.suffix+".part")
        if temp.exists():temp.unlink()
        h=hashlib.sha256();total=0;previous=0
        req=urllib.request.Request(base_url+name,headers={"User-Agent":"NobleAI-ColabQA/1.3"})
        with urllib.request.urlopen(req,timeout=90) as source,temp.open("wb") as target:
            while True:
                block=source.read(4*1024*1024)
                if not block:break
                target.write(block);h.update(block);total+=len(block)
                if total-previous>=32*1024*1024:
                    print("  descargados",round(total/1048576,1),"MiB de",name,flush=True)
                    previous=total
        if h.hexdigest()!=expected_sha:
            temp.unlink(missing_ok=True)
            raise RuntimeError("SHA256 incorrecto en "+name)
        temp.replace(dest)
    print("SHA256 verificado:",name,flush=True)
    extracted=root/(name.removesuffix(".tar.gz"))
    if not extracted.exists():
        extracted.mkdir(parents=True)
        print("Extrayendo",name,flush=True)
        with tarfile.open(dest,"r:gz") as archive:
            archive.extractall(extracted,filter="data")
    # Copy shared libraries into a common directory. The official archives
    # may contain nested build/bin and lib directories.
    library_count=0
    for path in extracted.rglob("*"):
        if path.is_file() and ".so" in path.name and path.name.startswith("lib"):
            shutil.copy2(path,bin_dir/path.name)
            library_count+=1
    print("Bibliotecas de",name,":",library_count,flush=True)
    for path in extracted.rglob("llama-server"):
        if path.is_file():
            shutil.copy2(path,bin_dir/"llama-server")
            (bin_dir/"llama-server").chmod(0o755)
binary=bin_dir/"llama-server"
if not binary.is_file():
    raise RuntimeError("Los archivos descargados no incluyen llama-server")
print("BINARIO LISTO:",binary,flush=True)
'''
        if (binaries_dir/"llama-server").is_file():
            compiled = True
            update("llama_prebuilt","PASS","Se reutiliza llama-server descargado anteriormente")
        else:
            compiled = command("llama_prebuilt",
                [sys.executable,"-u","-c",installer,str(BASE/"prebuilt_assets"),str(binaries_dir)],
                timeout=1000,watch=BASE/"prebuilt_assets")
        if compiled:
            import subprocess as _subprocess
            # The loader must know where libraries are BEFORE calling ldd.
            env_binary=os.environ.copy()
            env_binary["LD_LIBRARY_PATH"]=str(binaries_dir)+":"+env_binary.get("LD_LIBRARY_PATH","")
            probe=_subprocess.run(["ldd",str(binaries_dir/"llama-server")],
                                  capture_output=True,text=True,timeout=20,env=env_binary)
            (LOGS/"llama_binary_dependencies.log").write_text(
                clean(probe.stdout+"\n"+probe.stderr),encoding="utf-8")
            if probe.returncode != 0 or "not found" in probe.stdout:
                compiled = False
                update("llama_linker","FAIL",
                       "Bibliotecas incompatibles o ausentes; revisar llama_binary_dependencies.log")
            else:
                update("llama_linker","PASS","Bibliotecas dinámicas presentes")
                os.environ.update({"LD_LIBRARY_PATH":env_binary["LD_LIBRARY_PATH"]})
        if not compiled:
            skip("gpu_benchmark","No se pudo instalar o cargar llama.cpp precompilado; revisa los logs del descargador")
        else:
            hf = command("hf_package",
                 [sys.executable,"-m","pip","install","huggingface_hub>=0.27,<2"],timeout=480)
            if not hf:
                skip("gpu_benchmark","No se pudo instalar huggingface_hub")
            else:
                model_dir = BASE/"modelos"
                model_dir.mkdir(parents=True,exist_ok=True)
                meta = EVIDENCE/"modelo.json"
                # This repository tool validates real Hugging Face file names
                # and records the exact model revision and SHA256.
                fetched = command("model_download",
                    [sys.executable,"-u",str(ROOT/"tools/colab_hf_model.py"),
                     MODEL_REPO,MODEL_FILE,str(model_dir),str(meta)],
                    timeout=2100,watch=model_dir)
                if not fetched or not meta.exists():
                    skip("gpu_benchmark","No terminó la descarga GGUF")
                else:
                    info = json.loads(meta.read_text())
                    state["model"] = {k:v for k,v in info.items() if k!="path"}
                    save_status()
                    server = engine_repo/"build"/"bin"/"llama-server"
                    llm_log = (LOGS/"llama_server.log").open("ab",buffering=0)
                    children["llm"] = subprocess.Popen(
                        [str(server),"--model",info["path"],"--host","127.0.0.1",
                         "--port",str(LLM_PORT),"--ctx-size","4096","--n-gpu-layers","99",
                         "--parallel","1"],cwd=str(ROOT),
                        stdout=llm_log,stderr=subprocess.STDOUT,
                        stdin=subprocess.DEVNULL,start_new_session=True)
                    update("model_load","RUNNING","Cargando GGUF: máximo 180s")
                    available = False
                    model_id = ""
                    for attempt in range(180):
                        try:
                            models = http_json(f"http://127.0.0.1:{LLM_PORT}/v1/models",timeout=3)
                            model_id = models["data"][0]["id"]
                            available = True
                            break
                        except Exception: pass
                        if children["llm"].poll() is not None: break
                        if attempt%12==11:
                            print(f"   [ACTIVO] Cargando GGUF {attempt+1}s; GPU: {gpu_snapshot()[:1]}",flush=True)
                        time.sleep(1)
                    update("model_load","PASS" if available else "FAIL",
                           "Motor listo: "+MODEL_FILE if available else "Sin respuesta: revisa llama_server.log")
                    if not available:
                        skip("gpu_benchmark","El motor no logró cargar GGUF")
                    else:
                        config=http_json(f"http://127.0.0.1:{API_PORT}/api/v1/narrative/config")
                        config.update({"provider":"koboldcpp",
                            "base_url":f"http://127.0.0.1:{LLM_PORT}",
                            "model":model_id,"temperature":0.6,"max_tokens":150})
                        http_json(f"http://127.0.0.1:{API_PORT}/api/v1/narrative/config",
                                  config,method="PUT")
                        story = http_json(f"http://127.0.0.1:{API_PORT}/api/v1/stories",
                            {"title":"Diagnóstico de Colab",
                             "premise":"Una exploradora entra en una biblioteca encantada bajo la lluvia.",
                             "protagonist_name":"Nora"},method="POST")
                        story_id = story["id"]
                        update("narrator_connected","PASS","LLM conectado; historia de laboratorio creada")
                        from urllib.request import Request,urlopen
                        import uuid,statistics
                        records=[];samples=[];stop=threading.Event()
                        def observe_gpu():
                            while not stop.wait(.8):
                                samples.extend(gpu_snapshot())
                        watcher=threading.Thread(target=observe_gpu,daemon=True)
                        watcher.start()
                        try:
                            for n in range(max(1,min(10,RUNS))):
                                name="generation_"+str(n+1)
                                update(name,"RUNNING",f"Turno real {n+1}/{RUNS}, timeout 180s")
                                payload={"mode":"do" if n==0 else "continue",
                                     "input_text":"Nora abre un libro y examina sus páginas." if n==0 else "",
                                     "request_id":str(uuid.uuid4())}
                                start=time.monotonic()
                                first=None;characters=0;done=False;errors=[]
                                stopped=threading.Event()
                                def progress_inference():
                                    while not stopped.wait(12):
                                        print(f"   [ACTIVO] {name}: {int(time.monotonic()-start)}s; GPU: {gpu_snapshot()[:1]}",flush=True)
                                        state["stages"][name]["running_seconds"]=round(time.monotonic()-start,1)
                                        save_status()
                                pulse=threading.Thread(target=progress_inference,daemon=True)
                                pulse.start()
                                try:
                                    request=Request(
                                        f"http://127.0.0.1:{API_PORT}/api/v1/stories/{story_id}/generate",
                                        data=json.dumps(payload).encode(),method="POST",
                                        headers={"Content-Type":"application/json"})
                                    with urlopen(request,timeout=180) as response:
                                        for raw in response:
                                            line=raw.decode(errors="replace").strip()
                                            if not line.startswith("data: "):continue
                                            try: data=json.loads(line[6:])
                                            except ValueError: continue
                                            if "text" in data:
                                                if first is None:
                                                    first=round((time.monotonic()-start)*1000,1)
                                                characters+=len(data["text"])
                                            if "turn" in data:done=True
                                            if "message" in data:errors.append(str(data["message"])[:120])
                                    record={"success":done and characters>0,
                                            "ttft_ms":first,
                                            "duration_ms":round((time.monotonic()-start)*1000,1),
                                            "characters":characters,"errors":errors}
                                except Exception as exc:
                                    record={"success":False,"ttft_ms":first,
                                            "duration_ms":round((time.monotonic()-start)*1000,1),
                                            "characters":characters,"errors":[type(exc).__name__]}
                                finally:
                                    stopped.set();pulse.join(timeout=3)
                                records.append(record)
                                update(name,"PASS" if record["success"] else "FAIL",
                                       f"TTFT: {first}ms | Duración: {record['duration_ms']}ms | {characters} caracteres")
                        finally:
                            stop.set();watcher.join(timeout=3)
                        good=sorted(r["ttft_ms"] for r in records if r["success"] and r["ttft_ms"] is not None)
                        def quantile(values,p):
                            if not values:return None
                            index=(len(values)-1)*p;low=int(index);high=min(low+1,len(values)-1)
                            return round(values[low]+(values[high]-values[low])*(index-low),1)
                        state["benchmarks"]["gguf_real"]={
                            "runs":records,"completed":sum(int(r["success"]) for r in records),
                            "requested":RUNS,"ttft_p50_ms":quantile(good,.5),
                            "ttft_p95_ms":quantile(good,.95),
                            "vram_peak_observed_mb":max([g["used_mb"] for g in samples],default=None),
                            "gpu_util_peak_observed":max([g["util_percent"] for g in samples],default=None),
                            "notes":"VRAM muestreada, no pico exacto; sin medir imágenes ni TTS."}
                        GPU_RESULT=bool(records) and all(r["success"] for r in records)
                        update("gpu_benchmark","PASS" if GPU_RESULT else "FAIL",
                               f"Generaciones reales confirmadas {sum(r['success'] for r in records)}/{RUNS}")
    except Exception as exc:
        update("gpu_benchmark","FAIL",type(exc).__name__+": "+str(exc)[:210])
        (LOGS/"gpu_exception.log").write_text(traceback.format_exc()[-8000:],encoding="utf-8")
save_status()

## 6. Exportar ZIP y recuperar un informe parcial

Puedes ejecutar esta celda por separado si detuviste una etapa. Incluye logs y resultados existentes, sin exportar el GGUF ni los logs brutos del motor que podrían contener fragmentos narrativos.

In [ ]:
import json,zipfile,datetime
from pathlib import Path
folder=Path("/content/noble_qa_v14/diagnostico")
folder.mkdir(parents=True,exist_ok=True)
target=Path("/content/Noble_AI_Colab_QA_v1_4.zip")
if "state" in globals():
    report=state
else:
    old=folder/"report.json"
    report=json.loads(old.read_text()) if old.exists() else {"stages":{}}
report["packaged_utc"]=datetime.datetime.now(datetime.timezone.utc).isoformat()
report["stage_counts"]={label:sum(v.get("status")==label for v in report.get("stages",{}).values())
                        for label in ("PASS","FAIL","SKIP","RUNNING")}
(folder/"report.json").write_text(json.dumps(report,indent=2,ensure_ascii=False)+"\n",encoding="utf-8")
lines=["# Noble AI — Informe Colab QA v1.4","",
       "Commit: "+str(report.get("environment",{}).get("commit","desconocido")),
       "","## Etapas","","| Etapa | Estado | Detalle |","|---|---|---|"]
for stage,item in report.get("stages",{}).items():
    lines.append("| "+stage+" | "+item.get("status","?")+" | "+
                 str(item.get("detail","")).replace("|","/").replace("\n"," ")+" |")
lines.extend(["","## Hardware y benchmarks","",
              json.dumps({"environment":report.get("environment",{}),
                          "benchmarks":report.get("benchmarks",{})},ensure_ascii=False,indent=2),
              "","Solo las etapas PASS certifican su prueba. Imagen y TTS permanecen pendientes."])
(folder/"RESUMEN.md").write_text("\n".join(lines)+"\n",encoding="utf-8")
if "KEEP_APP_RUNNING" in globals() and not KEEP_APP_RUNNING and "children" in globals():
    for p in children.values():
        if p.poll() is None:p.terminate()
with zipfile.ZipFile(target,"w",compression=zipfile.ZIP_DEFLATED) as archive:
    for file in folder.rglob("*"):
        if not file.is_file() or file.stat().st_size>15000000:continue
        if file.name in ("llama_server.log","noble_api.log"):continue
        archive.write(file,arcname=str(file.relative_to(folder)))
print("Informe creado:",target,"|",round(target.stat().st_size/1024,1),"KB",flush=True)
print("Estados:",report["stage_counts"],flush=True)
try:
    from google.colab import files
    files.download(str(target))
except Exception as exc:
    print("Descarga manual: busca el ZIP en /content desde el explorador de Colab.",type(exc).__name__)

## 7. Vista previa opcional

La aplicación permanece en localhost; su interfaz puede abrirse con el visor interno de Colab.

In [ ]:
if globals().get("KEEP_APP_RUNNING") and globals().get("API_READY"):
    try:
        from google.colab import output
        output.serve_kernel_port_as_iframe(API_PORT,height=720)
    except Exception as exc:
        print("Vista previa no disponible:",type(exc).__name__,str(exc)[:160])
else:
    print("Vista previa no disponible o desactivada.")